##### Master Degree in Computer Science and Data Science for Economics

# Prompt Engineering

### Alfio Ferrara

For an introduction see
> Schulhoff, S., Ilie, M., Balepur, N., Kahadze, K., Liu, A., Si, C., ... & Resnik, P. (2024). The prompt report: A systematic survey of prompting techniques. arXiv preprint arXiv:2406.06608, 5.

## Options to interact with LLMs
-  Cloud APIs (e.g., OpenAI, Anthropic)
    - They are tipically easy-to-use but not always free. Data are processed online in cloud.
- Hugging Face Transformers (or other libraries)
    - Multiple models are available, and provides a complete control over the pipeline. Howevere everything is executed locally and a GPU is highly recommended.
- Runtime interfaces
    - There are multiple options, such as [llama.cpp](https://github.com/ggml-org/llama.cpp), [ollama](https://ollama.com/), [vllm](https://github.com/vllm-project/vllm)
    - They support multiple models and both local and remote execution

### A first example (using huggingface)

Sometimes a authentication token is required. See [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [1]:
import json
from huggingface_hub import login

In [2]:
with open('/Users/Flint/Data/apikeys/keys.json', 'r') as infile:
    token = json.load(infile)['huggingface']

login(token=token)

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

torch.set_default_device("mps")

model_id = "HuggingFaceH4/zephyr-7b-alpha"

print("Loading model... This may take a few minutes the first time.")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)  # MPS does not support float16

prompt = "<|system|>You are a helpful assistant.<|user|>What is prompt engineering?<|assistant|>"

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to("mps") for k, v in inputs.items()}  # Sposta tutto su MPS

print("Generating...")
outputs = model.generate(**inputs, max_new_tokens=200)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nZephyr's response:\n")
print(response)


Loading model... This may take a few minutes the first time.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generating...

Zephyr's response:

<|system|>You are a helpful assistant.<|user|>What is prompt engineering?<|assistant|>Prompt engineering is the process of designing and optimizing input prompts for natural language processing (NLP) models to improve their performance and accuracy. This involves selecting the right language, syntax, and structure for the prompts to ensure that the model can understand and respond appropriately to the input. Prompt engineering can also involve the use of pre-trained language models, such as GPT-3, to generate more complex and nuanced prompts that can better capture the intended meaning of the input. Overall, prompt engineering is a critical component of NLP development and can significantly improve the performance and usability of NLP models in various applications.


## Using APIs

In [4]:
from openai import OpenAI

with open('/Users/Flint/Data/apikeys/keys.json', 'r') as infile:
    apikey = json.load(infile)['openai']

client = OpenAI(api_key=apikey)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is prompt engineering in simple terms?"}
]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    max_tokens=300,
    temperature=0.7
)

print(response.choices[0].message.content)

Prompt engineering is the process of designing and creating prompts (such as questions, cues, or suggestions) to guide a user's behavior or decision-making towards a desired outcome. It involves strategically crafting these prompts to influence user actions in a specific way, often used in fields like user experience design, marketing, and behavior change interventions.


## Instruction + context + output format

In [5]:
def askgpt(messages, temperature=0.7):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature)
    return response.choices[0].message.content

**Generic and non-informative prompt**

In [6]:
messages = [
    {"role": "user", "content": "Write a recipe."}
]
answer = askgpt(messages)

print(answer)

Homemade Chicken Alfredo Pasta

Ingredients:
- 1 lb chicken breast, cut into bite-sized pieces
- 1 lb fettuccine pasta
- 2 cups heavy cream
- 1 cup grated Parmesan cheese
- 4 cloves garlic, minced
- 1/2 cup unsalted butter
- Salt and pepper to taste
- Fresh parsley, chopped (for garnish)

Instructions:
1. Cook the fettuccine pasta according to package instructions. Drain and set aside.
2. In a large skillet, melt the butter over medium heat. Add the minced garlic and cook until fragrant.
3. Add the chicken pieces to the skillet and cook until no longer pink, about 5-7 minutes.
4. Pour in the heavy cream and bring to a simmer. Let it cook for a few minutes until it thickens slightly.
5. Stir in the grated Parmesan cheese until melted and well combined.
6. Season with salt and pepper to taste.
7. Add the cooked fettuccine pasta to the skillet and toss to coat with the sauce.
8. Serve hot, garnished with chopped parsley.

Enjoy your delicious homemade Chicken Alfredo Pasta!


**Instructions**

Let's add some more instructions on the kind of language we want to be used. Still unclear what kind of recipe we want to get and how formatted.

In [7]:
messages = [
    {"role": "system", "content": "You are a professional chef."},
    {"role": "user", "content": "Write a recipe."}
]

answer = askgpt(messages)
print(answer)

**Recipe: Grilled Lemon Herb Chicken**

*Serves: 4*

**Ingredients:**
- 4 boneless, skinless chicken breasts
- 2 lemons
- 3 cloves of garlic, minced
- 2 tablespoons of fresh rosemary, chopped
- 2 tablespoons of fresh thyme, chopped
- 1/4 cup of olive oil
- Salt and pepper to taste

**Instructions:**

1. In a small bowl, combine the juice of 1 lemon, minced garlic, chopped rosemary, chopped thyme, olive oil, salt, and pepper. Mix well to create a marinade.

2. Place the chicken breasts in a shallow dish and pour the marinade over them, making sure to coat each piece evenly. Cover the dish and let the chicken marinate in the refrigerator for at least 1 hour (or overnight for best results).

3. Preheat your grill to medium-high heat.

4. Remove the chicken from the marinade and discard any excess marinade. Season the chicken with additional salt and pepper if desired.

5. Grill the chicken breasts for about 6-7 minutes per side, or until they are cooked through and have nice grill marks.


**Instructions + Context**

We want to add someting more about the kind of recipe we want to obtain

In [8]:
messages = [
    {"role": "system", "content": "You are writing recipes for a magazine that publishes easy-to-do recipes with few ingredients."},
    {"role": "user", "content": "Write a chinese recipe that an non experienced teenager can follow with easy-to-find ingredients."}
]

answer = askgpt(messages)
print(answer)

**Simple Stir-Fried Chicken with Vegetables**

*Ingredients:*
- 1 lb boneless, skinless chicken breast, cut into bite-sized pieces
- 2 cups mixed vegetables (such as bell peppers, broccoli, carrots)
- 2 cloves garlic, minced
- 2 tbsp soy sauce
- 1 tbsp oyster sauce
- 1 tbsp vegetable oil
- Salt and pepper, to taste
- Cooked rice, for serving

*Instructions:*
1. Heat vegetable oil in a large skillet or wok over medium-high heat.
2. Add the minced garlic and cook for about 30 seconds until fragrant.
3. Add the chicken pieces to the skillet and cook until no longer pink, about 5-7 minutes.
4. Add the mixed vegetables to the skillet and stir-fry for another 3-4 minutes until they are tender-crisp.
5. In a small bowl, mix together the soy sauce and oyster sauce. Pour the sauce over the chicken and vegetables in the skillet.
6. Stir everything together and cook for another 1-2 minutes until the sauce thickens slightly.
7. Season with salt and pepper to taste.
8. Serve the stir-fried chicken 

**Instructions + Context + output format**

We try to control the output

In [9]:
messages = [
    {"role": "system", "content": "You are writing recipes for a magazine that publishes easy-to-do recipes with few ingredients."},
    {"role": "user", "content": """
    Write a chinese recipe that an non experienced teenager can follow with easy-to-find ingredients.
    Provide the answer in json format like this: {"ingredients": [list of ingredients]}.
    Do not add anything but the ingredients. No title, no description and no comments!
    """}
]

answer = askgpt(messages)
print(answer)


{"ingredients": ["1 lb boneless chicken breast, cut into bite-sized pieces", "2 tablespoons soy sauce", "1 tablespoon honey", "1 tablespoon vegetable oil", "2 cloves garlic, minced", "1 teaspoon ginger, grated", "1 red bell pepper, sliced", "1 cup broccoli florets", "Cooked rice, for serving"]}


**Note about formatting**

This kind of formatting still requires parsing the json structure. With some libraries, like `llama.cpp` we can define the output using a `grammar`.

See more information on [GGML](https://github.com/ggml-org/llama.cpp/blob/master/grammars/README.md) website.

## Zero-shot, one-show, few-shot

1. **Zero-shot prompting**

    You ask the model to perform a task without showing any examples. It relies only on the instruction and prior training.

2. **One-shot prompting**

    You provide a single example of the task you want done, to guide the model.

3. **Few-shot prompting**

    You provide a few (2–5) examples to demonstrate the pattern or structure you're expecting.

#### Zero-shot

In [10]:
messages = [
    {"role": "system", "content": "You are a professional chef."},
    {"role": "user", "content": "Provide the ingredients of an italian recipe formatted as a simple list"}
]

answer = askgpt(messages)
print(answer)

Sure, here are the ingredients for a classic Italian pasta dish, Spaghetti Aglio e Olio:

- 8 oz spaghetti
- 4 cloves garlic, thinly sliced
- 1/4 cup extra virgin olive oil
- 1/2 tsp red pepper flakes
- Salt and pepper to taste
- Fresh parsley, chopped (for garnish)
- Grated Parmesan cheese (optional)


#### One- and few-shot 

In [11]:
messages = [
    {"role": "system", "content": "You are a professional chef."},
    {"role": "user", "content": """Provide the ingredients of an italian recipe formatted as a simple list
    An example is: "Title: Pasta with tomato and onions"
    "Ingredients": [pasta, olive oil, tomato sauce, onions]
     I want just the title and the ingredients and not the quantities!
    """}
]

answer = askgpt(messages)
print(answer)

Title: Classic Margherita Pizza
Ingredients: [pizza dough, fresh mozzarella, tomato sauce, fresh basil leaves, olive oil]


## Mode advanced methods

In advanced prompt engineering, the goal is not just to get an answer from the model, but to shape how the model thinks in order to achieve more consistent, 
accurate, and explainable outputs.

At this level, we move from “instructing” the model to designing reasoning patterns, i.e. ways of structuring prompts so that 
the model behaves predictably even in complex or ambiguous scenarios.
Effective prompts act like programs for reasoning: they define inputs, constraints, intermediate steps, and desired outputs.

Advanced techniques include:

- Chain-of-Thought prompting — guiding the model to reason step by step.
- Self-Consistency prompting — sampling multiple reasoning paths and comparing results.
- Reflexive or self-critique prompts — asking the model to evaluate or refine its own output.
- Tool-augmented prompting — designing prompts that invoke external processes (like search, code, or data transformation).
- Meta-prompts — prompts that generate or optimize other prompts.

Ultimately, advanced prompt engineering is about meta-control: understanding how to design structured interactions that produce not just creative answers, 
but reproducible reasoning.

## Prompt chain-of-thought

In [12]:
messages = [
    {"role": "system", "content": "You are a cousine espert."},
    {"role": "user", "content": """
    Provide a review on the pro and cons of the Italian cousine. Follow this scheme:
    1) Introduce the Italian cousine in a single sentence; 2) Summarize the main three pros; 3) summarize the main three cons;
    4) Conclude with an educated suggestion
    """}
]

answer = askgpt(messages)
print(answer)

1) Italian cuisine is renowned for its simple yet flavorful dishes that highlight fresh, high-quality ingredients.

2) Pros:
   a) Fresh Ingredients: Italian cuisine emphasizes the use of fresh, seasonal ingredients, resulting in vibrant flavors and nutritious meals.
   b) Diverse Regional Flavors: Italy's diverse regions offer a wide range of culinary traditions, from rich pasta dishes in the North to fresh seafood along the coast.
   c) Versatility: Italian cuisine offers a variety of options for different dietary preferences, including vegetarian, gluten-free, and vegan dishes.

3) Cons:
   a) Heavy on Carbs: Traditional Italian dishes often contain a lot of carbohydrates, which may not be suitable for those following a low-carb diet.
   b) Time-consuming Preparation: Some Italian recipes require lengthy preparation and cooking times, making them less practical for busy weeknights.
   c) High in Calories: Many Italian dishes are rich in fats and calories, which may not align with he

### Prompt with constraints concerning style, lenght, format, etc.

In [16]:
messages = [
    {"role": "system", "content": "You are a cousine espert."},
    {"role": "user", "content": """
    Write a short Italian recipe in the style of Lord Byron
    """}
]

answer = askgpt(messages)
print(answer)

Ah, how the pasta doth twine, in a dance divine,
With garlic, basil, and olive oil so fine.
In a pan, the tomatoes shall gently simmer,
As the aromas waft, our senses do glimmer.

Al dente spaghetti, a touch of sea salt,
A symphony of flavors, a feast without fault.
With Parmesan grated, a sprinkle of gold,
This dish of simplicity, a treasure untold.

Serve with a flourish, a glass of red wine,
Let us toast to Italy, where the sun doth shine.
A culinary delight, a joy to savor,
In this recipe divine, let us revel and favor.


In [13]:
def chat(content, system="useful assistant"):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": content}
    ]
    answer = askgpt(messages)
    return answer.replace(". ", ".\n")

### Chain-of-Thought prompting

In [14]:
naive_prompt = """
Estimate how many cousine chef there are in Paris.
"""
cot_prompt = "\n ".join([
    'Estimate the population of Paris', 'Estimate the number of restaurants in Paris',
    'Take into account that typically each restaurant needs a chef or two',
    'Calculate the number of chef needed',
    'At the end, give your final estimate and explain your reasoning briefly.'
])

print(f"Naive prompt: {chat(content=naive_prompt)}\n\n")
print(f"CoT prompt: {chat(content=cot_prompt)}")

Naive prompt: It is difficult to provide an exact number of cuisine chefs in Paris, as there are many chefs working in various types of cuisine and at different levels of expertise.
However, it is estimated that there are thousands of cuisine chefs working in the many restaurants, cafes, bakeries, and other food establishments in Paris.
The city is known for its culinary excellence and diverse dining options, so the number of cuisine chefs is likely to be quite high.


CoT prompt: To estimate the population of Paris, we can refer to the most recent data available.
As of 2021, the population of Paris is estimated to be around 2.2 million people.

Next, let's estimate the number of restaurants in Paris.
According to data, Paris is known for its vibrant culinary scene and is home to a large number of restaurants.
As of 2021, there are approximately 18,000 restaurants in Paris.

Considering that each restaurant typically needs at least one chef, and some larger restaurants may require two 

### Self-evaluation & Iterative Improvement

In [15]:
naive_prompt = """
Write a 200-word summary of the main features of Italian Cousine.
"""
answer = chat(content=naive_prompt)
print(f"Naive prompt: {answer}\n\n")

self_prompt = f"This is a short summary of the italian cousine features: {chat(content=naive_prompt)}. "
self_prompt += "Critique this summary as if you were a cousine expert reviewing it for a publications. Identify at least two weaknesses. "
self_prompt += "Rewrite the summary incorporating your own feedback"

eh_answer = chat(content=self_prompt, system="you are a cousine expert")
print(f"Self support prompt: {eh_answer}\n\n")



Naive prompt: Italian cuisine is known for its rich flavors, fresh ingredients, and regional diversity.
One of the main features of Italian cuisine is its emphasis on simplicity and quality of ingredients.
Italian dishes typically use fresh, local ingredients such as olive oil, tomatoes, garlic, herbs, and cheese.

Pasta is a staple in Italian cuisine, with a wide variety of shapes and sauces to choose from.
Some popular pasta dishes include spaghetti carbonara, fettuccine alfredo, and lasagna.
Italian cuisine also features a wide range of risottos, pizzas, and antipasti (appetizers) such as bruschetta and caprese salad.

Italian cuisine is also known for its delicious desserts, including tiramisu, cannoli, and gelato.
Wine is an important part of Italian dining, with each region producing its own unique varieties.

Overall, Italian cuisine is characterized by its use of fresh, high-quality ingredients, simple preparation methods, and emphasis on regional specialties.
It is a cuisine t